In [ ]:
"""
count_word_and_trigram_frequencies_kaggle.py
Count word and trigram frequencies from wikitext_train.txt on Kaggle
"""

import re
from collections import Counter
from nltk.util import ngrams
import pickle
import os

def count_word_and_trigram_frequencies(input_file, word_output_file, trigram_output_file):
    """
    Count word frequencies and trigram frequencies in the training file.
    
    Args:
        input_file: Full path to wikitext_train.txt
        word_output_file: Output pickle file for word->frequency mapping
        trigram_output_file: Output pickle file for trigram->frequency mapping
    """
    print("🔍 Counting word and trigram frequencies from raw text...")

    word_counts = Counter()
    trigram_counts = Counter()
    total_words = 0
    line_count = 0
    prev_words = []  # for trigrams spanning lines

    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            line_count += 1

            # Clean and tokenize the line into words
            line = line.strip()
            if not line:
                continue

            words = re.findall(r'\b[\w\']+\b', line.lower())

            # Update word counts
            word_counts.update(words)
            total_words += len(words)

            # Prepare for trigram counts
            if prev_words:
                words = prev_words + words
            if len(words) >= 3:
                trigram_counts.update(ngrams(words, 3))
            prev_words = words[-2:]  # keep last 2 words for next line

            # Progress tracking
            if line_count % 10000 == 0:
                print(f"  Processed {line_count:,} lines... ({total_words:,} words)")

    print(f"📊 Finished processing!")
    print(f"📈 Total lines processed: {line_count:,}")
    print(f"📈 Total words counted: {total_words:,}")
    print(f"📊 Unique words found: {len(word_counts):,}")
    print(f"📊 Unique trigrams found: {len(trigram_counts):,}")

    # Save as pickle files
    with open(word_output_file, "wb") as f:
        pickle.dump(word_counts, f)
    print(f"💾 Word frequencies saved as: {word_output_file}")

    with open(trigram_output_file, "wb") as f:
        pickle.dump(trigram_counts, f)
    print(f"💾 Trigram frequencies saved as: {trigram_output_file}")

    return word_counts, trigram_counts, total_words

def main():
    # -------------------------------
    # Kaggle paths
    # -------------------------------
    INPUT_FILE = "/kaggle/input/frequency-2-0/wikitext_train.txt"  # correct input path
    WORD_PKL = "/kaggle/working/word_frequencies.pkl"             # writable output
    TRIGRAM_PKL = "/kaggle/working/trigram_frequencies.pkl"

    # Optional: check file exists
    if not os.path.exists(INPUT_FILE):
        raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

    print("=" * 60)
    print("📚 Word & Trigram Frequency Counter (Kaggle)")
    print("=" * 60)

    # Count word and trigram frequencies
    word_counts, trigram_counts, total_words = count_word_and_trigram_frequencies(
        INPUT_FILE, WORD_PKL, TRIGRAM_PKL
    )

    # Optional: show top 20 most frequent words
    print(f"\n🏆 Top 20 words:")
    for i, (word, count) in enumerate(word_counts.most_common(20), 1):
        print(f"  {i:2d}. {word:15} - {count:,}")

    # Optional: show top 10 most frequent trigrams
    print(f"\n🏆 Top 10 trigrams:")
    for i, (trigram, count) in enumerate(trigram_counts.most_common(10), 1):
        print(f"  {i:2d}. {' '.join(trigram)} - {count:,}")

    print("\n🎉 Done! Pickle files are ready in /kaggle/working/")

if __name__ == "__main__":
    main()


📚 Word & Trigram Frequency Counter (Kaggle)
🔍 Counting word and trigram frequencies from raw text...
📊 Finished processing!
📈 Total lines processed: 2,330,058
📈 Total words counted: 86,537,628
📊 Unique words found: 226,126
📊 Unique trigrams found: 42,564,445
💾 Word frequencies saved as: /kaggle/working/word_frequencies.pkl


In [9]:
import pickle

# Load the word frequencies
with open("/kaggle/working/word_frequencies.pkl", "rb") as f:
    word_freq = pickle.load(f)

# word_freq is a Counter object, so you can use it like a dictionary
print(f"Total unique words: {len(word_freq)}")

# Example: frequency of a specific word
word = "the"
print(f"'{word}' occurs {word_freq[word]} times")

# Example: top 10 most frequent words
print("\nTop 10 words:")
for w, count in word_freq.most_common(10):
    print(f"{w}: {count}")


Total unique words: 226126
'the' occurs 6438861 times

Top 10 words:
the: 6438861
of: 2743105
and: 2505742
in: 2176387
to: 1994950
a: 1787681
was: 1076118
s: 782817
on: 769184
as: 725415


In [11]:
import pickle
from collections import Counter

# Load the trigram frequencies
with open("/kaggle/working/trigram_frequencies.pkl", "rb") as f:
    trigram_freq = pickle.load(f)

# Function to predict next word using trigram model
def predict_next_word(sentence, trigram_counts, top_k=1):
    """
    Predict the next word given a sentence based on trigram frequencies.
    
    Args:
        sentence (str): Input sentence (can be incomplete)
        trigram_counts (Counter): Counter of all trigrams
        top_k (int): Number of top predictions to return

    Returns:
        List of predicted next words (most probable first)
    """
    import re
    # Clean and tokenize sentence
    words = re.findall(r'\b[\w\']+\b', sentence.lower())
    if len(words) < 2:
        print("Please provide at least two words.")
        return []

    # Consider last two words
    w1, w2 = words[-2], words[-1]

    # Find candidate next words
    candidates = {w3: count for (x1, x2, w3), count in trigram_counts.items() if x1 == w1 and x2 == w2}

    if not candidates:
        print("No trigram found for these last two words.")
        return []

    # Sort candidates by frequency
    sorted_candidates = sorted(candidates.items(), key=lambda item: item[1], reverse=True)

    # Return top_k predictions
    return [word for word, count in sorted_candidates[:top_k]]

# Example usage
sentence = "I am"
predictions = predict_next_word(sentence, trigram_freq, top_k=3)
print(f"Next word predictions for '{sentence}': {predictions}")


Next word predictions for 'I am': ['not', 'a', 'the']
